# Task 2 - Leakage-aware classification

Four classifiers are tuned using raw training rows and a Pipeline. Imputation and scaling
are refitted inside every cross-validation fold. A validation set selects the model family;
only that model and the most-frequent baseline are evaluated on the final test set.
CV tuning scores, validation scores and test scores have different roles.

Set `ML_LAB_SMOKE=1` for a reduced check. This mode is not a full benchmark.

In [ ]:
import json
import os
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

from ml_lab.experiments import load_dataset, benchmark

SMOKE = os.getenv("ML_LAB_SMOKE") == "1"
Path("results").mkdir(exist_ok=True)
reports = {}
for dataset in ("diabetes", "digits"):
    report = benchmark(load_dataset(dataset, quick=SMOKE), quick=SMOKE)
    report["kind"] = "classical"
    reports[dataset] = report
    Path(f"results/task2_{dataset}.json").write_text(
        json.dumps(report, indent=2, allow_nan=False), encoding="utf-8")
    print(dataset, report["mode"], report["split_sizes"])
    print(pd.DataFrame(report["candidates"]).drop(columns="params").to_string(index=False))
    print("Selected:", report["selected_model"])
    print("Held-out accuracy:", report["test"]["accuracy"])
    print("Most-frequent baseline:", report["baseline_test"]["accuracy"])


## Final test confusion matrices

The test set was kept out of preprocessing fits, hyperparameter search and model-family
selection. The selected pipeline was refitted on train + validation before evaluation.
Repeatedly changing the experiment after reading this test result would also compromise
the holdout; use a new external test set for further development.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, (dataset, report) in zip(axes, reports.items()):
    matrix = report["test"]["confusion_matrix"]
    image = ax.imshow(matrix, cmap="Blues")
    for row in range(len(matrix)):
        for col in range(len(matrix)):
            ax.text(col, row, str(matrix[row][col]), ha="center", va="center")
    ax.set_title(f"{dataset}: {report['selected_model']} ({report['mode']})")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
    fig.colorbar(image, ax=ax)
fig.tight_layout()
fig.savefig("results/task2_confusion.png", dpi=120)
plt.show()


## Interpretation

Compare the selected model with the baseline and examine per-class precision/recall in
the saved JSON. There is no fixed winning model or claimed accuracy in this notebook.
The Diabetes example is an educational dataset exercise and is not a diagnostic system.
Reports include the seed, split sizes, searched configurations and package versions.